In [1]:
# --- Cell 1 : interactive inputs ---------------------------------
import ipywidgets as widgets
from ipywidgets import interact
from datetime import date as _date

lat_widget  = widgets.FloatText(value=-27.47711,
                                description='Latitude (°)',
                                step=0.0001)
lon_widget  = widgets.FloatText(value=153.029342,
                                description='Longitude (°)',
                                step=0.0001)
date_widget = widgets.DatePicker(description='Date',
                                 value=_date(2025, 8, 4))

print("Enter your observing location and date, then run the next cell.")
display(lat_widget, lon_widget, date_widget)

Enter your observing location and date, then run the next cell.


FloatText(value=-27.47711, description='Latitude (°)', step=0.0001)

FloatText(value=153.029342, description='Longitude (°)', step=0.0001)

DatePicker(value=datetime.date(2025, 8, 4), description='Date', step=1)

In [2]:
# --- Cell 2 : calculation & table --------------------------------
import pandas as pd
from datetime import datetime, time, timedelta
from zoneinfo import ZoneInfo
from timezonefinder import TimezoneFinder
from astropy.coordinates import EarthLocation, get_sun, AltAz
from astropy.time import Time
import astropy.units as u

# Read values from widgets
latitude  = lat_widget.value
longitude = lon_widget.value
calculation_date = date_widget.value

# Look up the local time zone from the coordinates (falls back to UTC, e.g. at sea)
tz_name  = TimezoneFinder().timezone_at(lat=latitude, lng=longitude) or 'UTC'
TIMEZONE = ZoneInfo(tz_name)

def calculate_solar_elevation(lat, lon, date_of_interest):
    location = EarthLocation(lat=lat*u.deg, lon=lon*u.deg, height=0*u.m)
    results  = []
    current  = datetime.combine(date_of_interest, time(9, 0), tzinfo=TIMEZONE)
    end      = datetime.combine(date_of_interest, time(15, 0), tzinfo=TIMEZONE)
    interval = timedelta(minutes=30)
    tz_abbr  = current.tzname()  # e.g. AEST, AEDT, GMT

    while current <= end:
        t = Time(current)
        sun_altaz = get_sun(t).transform_to(AltAz(obstime=t,
                                                 location=location))
        results.append({
            f"Time ({tz_abbr})": current.strftime('%H:%M'),
            "Elevation (°)": f"{sun_altaz.alt.degree:.1f}"
        })
        current += interval

    return pd.DataFrame(results)

elevation_table = calculate_solar_elevation(latitude, longitude,
                                            calculation_date)

print(f"Solar Elevation for ({latitude:.4f}, {longitude:.4f})")
print(f"Date: {calculation_date.strftime('%d %B %Y')}")
print(f"Time zone: {tz_name}")
print("-" * 50)
print(elevation_table.to_string(index=False))

# I recommend confirming this against https://gml.noaa.gov/grad/solcalc/

Solar Elevation for (-27.4771, 153.0293)
Date: 04 August 2025
Time zone: Australia/Brisbane
--------------------------------------------------
Time (AEST) Elevation (°)
      09:00          28.5
      09:30          33.3
      10:00          37.4
      10:30          40.9
      11:00          43.4
      11:30          44.9
      12:00          45.3
      12:30          44.5
      13:00          42.6
      13:30          39.6
      14:00          35.9
      14:30          31.5
      15:00          26.6
